In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold

X = pd.read_csv("../data/processed/X_features.csv", index_col=0)
Y = pd.read_csv("../data/processed/response_delta.csv", index_col=0)
groups = pd.read_csv("../data/processed/row_groups.csv", index_col=0)["group"]
meta = pd.read_csv("../data/processed/response_meta.csv", index_col=0)

bits = [c for c in X.columns if c != "log_dose"]
dose = X["log_dose"].values
Yv = Y.values

In [11]:
X.head()

,bit1,bit2,bit3,bit4,bit5,bit6,bit7,bit8,bit9,bit10,...,bit1015,bit1016,bit1017,bit1018,bit1019,bit1020,bit1021,bit1022,bit1023,log_dose
product_dose,,,,,,,,,,,,,,,,,,,,,
Enzastaurin (LY317615)_1000,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,3.0
Raltitrexed_10,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.0
Lenalidomide (CC-5013)_10,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1.0
Divalproex Sodium_1000,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3.0
MLN8054_100,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,2.0


In [12]:
Y.head()

,GCLC,M6PR,KLHL13,CYP26B1,TFPI,MTMR7,PLXND1,PDK4,ARX,CALCR,...,AC139493.2,AL512380.2,FAM95C,MIR4721,AC006460.2,AC025287.4,AC092718.8,AC015802.6,AL033527.4,AL034417.3
Enzastaurin (LY317615)_1000,0.013655,-0.003842,-0.015394,0.004320,0.126866,-0.001997,-0.001670,0.004187,-0.006113,-0.017229,...,-0.002352,0.003411,-0.004409,-0.001046,0.016559,-0.000782,-0.002204,-0.010622,-0.000562,-0.004960
Raltitrexed_10,-0.082742,-0.026113,-0.004750,-0.001979,0.005126,-0.001997,-0.017176,0.035341,0.013636,-0.002438,...,-0.002352,-0.009185,0.024590,-0.001046,-0.009247,-0.000782,0.007587,0.001867,-0.000562,-0.009498
Lenalidomide (CC-5013)_10,-0.025102,0.038318,-0.003858,-0.013038,-0.042092,-0.001997,-0.040192,0.085220,-0.006113,-0.003623,...,-0.002352,-0.009185,-0.004409,0.004940,0.020209,-0.000782,0.012083,0.000546,0.007956,-0.009498
Divalproex Sodium_1000,0.258022,0.051005,0.035410,0.017052,0.141356,-0.001997,0.054104,0.093719,-0.006113,-0.000082,...,-0.002352,-0.009185,-0.004409,-0.001046,-0.028486,-0.000782,-0.002204,0.009121,-0.000562,-0.009498
MLN8054_100,0.114972,-0.000282,0.063457,0.000425,0.334283,-0.001997,-0.027579,0.057526,-0.006113,-0.017229,...,-0.002352,-0.002166,0.007701,-0.001046,0.055626,-0.000782,0.022978,-0.010622,-0.000562,0.007781


In [13]:
groups

product_dose
Enzastaurin (LY317615)_1000      64
Raltitrexed_10                  131
Lenalidomide (CC-5013)_10        92
Divalproex Sodium_1000           57
MLN8054_100                     100
                               ... 
Epothilone A_10                  65
PHA-680632_100                  120
Trametinib (GSK1120212)_10      168
Iniparib (BSI-201)_10000         84
Tucidinostat (Chidamide)_100    173
Name: group, Length: 750, dtype: int64

In [8]:
groups

product_dose
Enzastaurin (LY317615)_1000      64
Raltitrexed_10                  131
Lenalidomide (CC-5013)_10        92
Divalproex Sodium_1000           57
MLN8054_100                     100
                               ... 
Epothilone A_10                  65
PHA-680632_100                  120
Trametinib (GSK1120212)_10      168
Iniparib (BSI-201)_10000         84
Tucidinostat (Chidamide)_100    173
Name: group, Length: 750, dtype: int64

In [3]:
#Run the 5 folds, filling in predictions for the test rows
P_mean = np.zeros_like(Yv)
P_knn = np.zeros_like(Yv)
B = X[bits].values.astype(bool)

gkf = GroupKFold(n_splits=5)
for tr, te in gkf.split(X, Y, groups):
    # mean baseline: average of training rows at the same dose
    for d in np.unique(dose[te]):
        P_mean[te[dose[te] == d]] = Yv[tr[dose[tr] == d]].mean(axis=0)

    # nearest neighbour: most similar training drug at the same dose
    for i in te:
        sim = (B[i] & B[tr]).sum(1) / (B[i] | B[tr]).sum(1)
        sim[dose[tr] != dose[i]] = -1
        P_knn[i] = Yv[tr[sim.argmax()]]

In [4]:
#Score each row
def row_r(P, T):
    return np.array([np.corrcoef(p, t)[0, 1] for p, t in zip(P, T)])

scores = pd.DataFrame({
    "r_mean": row_r(P_mean, Yv),
    "r_knn": row_r(P_knn, Yv),
    "mse_mean": ((P_mean - Yv) ** 2).mean(axis=1),
    "mse_knn": ((P_knn - Yv) ** 2).mean(axis=1),
}, index=Y.index)
scores["responder"] = meta["responder"]

print("All rows:")
print(scores.drop(columns="responder").mean().round(4))
print("Responders only:")
print(scores[scores["responder"]].drop(columns="responder").mean().round(4))

All rows:
r_mean      0.2508
r_knn       0.1595
mse_mean    0.0038
mse_knn     0.0063
dtype: float64
Responders only:
r_mean      0.4229
r_knn       0.3334
mse_mean    0.0105
mse_knn     0.0120
dtype: float64


In [5]:
scores.to_csv("../results/tables/05_baseline_scores.csv")

In [14]:
scores

,r_mean,r_knn,mse_mean,mse_knn,responder
Enzastaurin (LY317615)_1000,-0.048482,-0.085715,0.001358,0.002464,False
Raltitrexed_10,0.168572,0.037277,0.000935,0.002161,False
Lenalidomide (CC-5013)_10,0.233470,0.115178,0.000727,0.001324,False
Divalproex Sodium_1000,0.352782,0.158360,0.001483,0.002288,False
MLN8054_100,0.504737,0.802585,0.006021,0.002685,True
...,...,...,...,...,...
Epothilone A_10,0.293821,0.815489,0.007824,0.003143,True
PHA-680632_100,0.423591,0.240589,0.001520,0.002236,False
Trametinib (GSK1120212)_10,0.277249,-0.273558,0.010531,0.013559,True
Iniparib (BSI-201)_10000,-0.005665,0.222864,0.005185,0.004280,True
